# ECOSIM-G v3 en Kaggle: miles de mundos en las dos T4, una fila por mundo

**ECOSIM-G v3.1** son las 100 mecánicas (supervivencia, economía, social, psicología, política, cultura y el mundo) escritas en Ñ para **C1EÑ**. Cada mundo tiene 2.000 plazas de personas, 6 territorios, 3 Estados, 3 religiones, hasta 90 empresas (las que fundan las personas caben en las plazas libres), 100 inversores y 3 bancos. Un tick es un segundo simulado y **un año son 3.600 ticks**; por defecto se corren 21.600 ticks (6 años).

La **3.1** arregla el estancamiento de la 3.0: sueldos por empresa y por rendimiento, personas que fundan empresas (con crédito), investigación en las empresas, la tecnología que se contagia entre Estados, herencias repartidas entre los hijos. El CSV trae 20 columnas nuevas para verlo (`fundadasPersonas`, `crecimientoReal`, `sueldoBase`, `pobresQueSalieron`, `ricosQueSalieron`…).

Lo que sale de aquí: **un solo CSV con una fila por mundo** (101 columnas: población, nacimientos, muertes, hambre, Hoover, guerras, conquistas, revoluciones, tecnología, bosque, bolsa…). Si pasa de 7 MB se reparte en partes de 7 MB como mucho, cada una con su encabezado (`ecosim-g3-mundos-1.csv`, `-2.csv`…). Unos 15.000 mundos caben en un archivo (una fila ocupa unos 465 bytes).

**Por qué hay lotes.** Un arreglo de Ñ tiene como mucho 50 millones de números, y los paquetes de las personas son `mundos × plazas × 16`: con 2.000 plazas caben **1.562 mundos por proceso**. Para miles de mundos se lanzan varios procesos (lotes de 1.500), repartidos entre las dos GPU. El mundo número `n` usa siempre la semilla `SEMILLA + n`, sea cual sea el lote donde cae: se comprobó que 6 mundos en un proceso y 3 + 3 en dos procesos dan el CSV idéntico, byte a byte.

**Cuántos caben.** Cada mundo ocupa ≈ 1,2 MB en la GPU. Una T4 tiene ~15 GB: por memoria cabrían ~12.000 mundos en cada una. El cuaderno corre 2 lotes a la vez por GPU (3,5 GB de cada T4; los 4 vCPU de Kaggle mandan sobre cuántos procesos conviene) y pone los demás en cola.

| mundos | lotes de 1.500 | CSV | tiempo estimado (6 años) |
|---|---|---|---|
| 3.000 | 2 | ~1,4 MB, 1 archivo | 3–6 min |
| 12.000 | 8 | ~5,6 MB, 1 archivo | 9–22 min |
| 24.000 | 16 | ~11 MB, 2 archivos | 18–45 min |
| 50.000 | 34 | ~23 MB, 4 archivos | 36–90 min |

Los tiempos son **una estimación sin medir en una T4** (≈ 4–10 ms por tick por cada 1.000 mundos y GPU). La celda 5 los mide de verdad: el log de cada lote dice cuántos ms tardó por tick.

**Antes de empezar:** *Settings → Accelerator → GPU T4 x2*. Para la opción B (clonar de GitHub), *Settings → Internet → On*.

In [ ]:
# 1 · Node 22 (Kaggle trae uno viejo o ninguno) — se instala en /kaggle/temp (no va al Output) sin tocar el sistema
import os, subprocess, shutil, re
import glob
os.makedirs('/kaggle/temp', exist_ok=True)   # lo pesado (Node, el motor de Ñ) va aquí: Kaggle no lo guarda en el Output
for v in [f'/kaggle/working/{x}' for x in ['n', 'lotes', 'prueba-serie.csv', 'prueba-mundos.csv', 'prueba.log']] + glob.glob('/kaggle/working/node-v*-linux-x64'):
    if os.path.isdir(v) and not os.path.islink(v): shutil.rmtree(v, ignore_errors=True); print('borrado del Output:', v)
    elif os.path.exists(v): os.remove(v); print('borrado del Output:', v)
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /kaggle/temp', shell=True, check=True)
    os.environ['PATH'] = f'/kaggle/temp/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())

In [ ]:
# 2 · El proyecto. Opción A: un dataset de Kaggle con la carpeta del proyecto (la que tiene motor/ dentro).
#     Opción B: git clone del repositorio. Se usa A si se encuentra; si no, B.
import glob
PROYECTO = '/kaggle/temp/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if os.path.exists(PROYECTO):
    print('ya estaba en', PROYECTO)
elif candidatos:
    shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules'))
    print('opción A: copiado desde', candidatos[0])
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/diegovelandiabarajas1-lang/Espanol-Like', PROYECTO], check=True)
    print('opción B: clonado en', PROYECTO)
os.chdir(PROYECTO)
assert os.path.exists('motor/src/14-c1en.js'), 'este proyecto no trae C1EÑ (motor/src/14-c1en.js)'
assert os.path.exists('ecosim/ecosim-g3.esl') and os.path.exists('ecosim/g3/mundo.esl'), 'falta ecosim/ecosim-g3.esl o ecosim/g3/: sube la versión del proyecto que los trae'

In [ ]:
# 3 · koffi (el FFI de Node que habla con libcuda y libnvrtc), el bundle del motor y las GPU
!cd /kaggle/temp/n && npm install koffi --no-audit --no-fund --loglevel=error && node motor/util/construir.js
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv
!cd /kaggle/temp/n && node -e "require('./motor/bundle.js'); const m = globalThis.EspanolLike.crearMotor({}); console.log(m.vm.c1en.disponibles().join('\n'))"

## Una corrida pequeña en la GPU 0

10 mundos × 1 año. Al final el programa dice si los invariantes (dinero con el tesoro de los Estados, deudas = préstamos, acciones, bienes con las despensas, participaciones) se cumplieron en todos los mundos, e imprime el resumen de las 100 mecánicas.

In [ ]:
import time
TRABAJO = '/kaggle/working'
def correr(mundos, ticks, plazas, semilla, primer, serie, csv_mundos, gpu, log, por_sector=6, credito=1):
    env = dict(os.environ, C1EN_DISPOSITIVO='gpu', C1EN_GPU=str(gpu), C1EN_ESTADISTICAS='1')
    cmd = ['node', 'motor/util/n-node.js', '--c1en', 'ecosim/ecosim-g3.esl', str(mundos), str(ticks), str(plazas),
           str(semilla), serie, str(por_sector), str(credito), str(primer), csv_mundos]
    return subprocess.Popen(cmd, env=env, stdout=open(log, 'w'), stderr=subprocess.STDOUT, text=True)

t0 = time.time()
p = correr(10, 3600, 2000, 20261018, 0, f'/kaggle/temp/prueba-serie.csv', f'/kaggle/temp/prueba-mundos.csv', gpu=0, log=f'/kaggle/temp/prueba.log')
p.wait()
print(open(f'/kaggle/temp/prueba.log').read())
print(f'{time.time() - t0:.1f} s en total (incluye compilar los núcleos con NVRTC)')

## Los parámetros: cuántos mundos

`MUNDOS` es el total. Se parten en lotes de `POR_LOTE` mundos (como mucho 1.562 con 2.000 plazas; con menos plazas caben más por lote: el máximo se calcula solo). Cada lote es un proceso con su `primerMundo`; se corren `A_LA_VEZ_POR_GPU` lotes a la vez en cada T4 y el resto espera en cola.

In [ ]:
MUNDOS = 12000            # en total: miles, o más
TICKS = 21600             # 6 años (un año = 3600 ticks)
PLAZAS = 2000             # personas por mundo como mucho
SEMILLA = 20261018        # el mundo n usa SEMILLA + n
POR_SECTOR = 6            # 6 × 5 sectores = 30 empresas al empezar, hasta 90
CREDITO = 1
A_LA_VEZ_POR_GPU = 2      # sube a 3 o 4 si los vCPU no van al 100 %
GPUS = [0, 1]

# el tope del motor: arreglos de 50 millones de números (igual que en ecosim-g3.esl)
N_INI = 5 * POR_SECTOR
N_EMP = N_INI * 3
TEN = 1 + 100 + N_EMP
MAX_POR_PROCESO = min(50_000_000 // (PLAZAS * 16), 50_000_000 // (N_EMP * TEN))
POR_LOTE = min(1500, MAX_POR_PROCESO)
lotes = [(i, a, min(POR_LOTE, MUNDOS - a)) for i, a in enumerate(range(0, MUNDOS, POR_LOTE))]

MB_MUNDO = (PLAZAS * 135 + N_EMP * TEN + N_EMP * 70) * 4 / 1e6    # ≈ 1,2 MB con 2000 plazas
libre = [int(x) for x in subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'],
                                        capture_output=True, text=True).stdout.split()]
ram = int(open('/proc/meminfo').read().split('MemAvailable:')[1].split()[0]) / 1e6
print(f'{MUNDOS} mundos en {len(lotes)} lotes de hasta {POR_LOTE} (máximo por proceso: {MAX_POR_PROCESO})')
print(f'cada mundo ≈ {MB_MUNDO:.2f} MB · cada lote ≈ {POR_LOTE * MB_MUNDO / 1000:.1f} GB en la GPU (y otro tanto en la RAM)')
print(f'a la vez: {A_LA_VEZ_POR_GPU} lotes por GPU → {A_LA_VEZ_POR_GPU * POR_LOTE * MB_MUNDO / 1000:.1f} GB de cada T4 (libres: {", ".join(f"{m / 1000:.1f}" for m in libre)} GB)')
print(f'RAM: {len(GPUS) * A_LA_VEZ_POR_GPU * POR_LOTE * MB_MUNDO / 1000:.1f} GB de {ram:.1f} GB disponibles')
print(f'por memoria de GPU cabrían ~{int(min(libre) * 0.9 / MB_MUNDO):,} mundos en cada T4')
assert A_LA_VEZ_POR_GPU * POR_LOTE * MB_MUNDO < min(libre) * 0.9, 'no cabe: baja A_LA_VEZ_POR_GPU o PLAZAS'
assert len(GPUS) * A_LA_VEZ_POR_GPU * POR_LOTE * MB_MUNDO / 1000 < ram * 0.85, 'no cabe en la RAM: baja A_LA_VEZ_POR_GPU'

## Correr todos los lotes

Una cola sencilla: cada GPU toma el siguiente lote cuando uno de los suyos termina. Cada lote deja en `/kaggle/temp/lotes/` (fuera del Output: solo el CSV juntado va al Output) su CSV de mundos, su serie (los 20 primeros mundos del lote, cada 10 minutos simulados) y su log. Si un lote falla, se ve el final de su log y se puede volver a correr solo ese (mismo `primerMundo`, mismos mundos).

In [ ]:
import collections
LOTES = '/kaggle/temp/lotes'      # los CSV de cada lote: intermedios, fuera del Output
os.makedirs(LOTES, exist_ok=True)
cola = collections.deque(lotes)
corriendo = {g: [] for g in GPUS}
hechos, fallidos = [], []
t0 = time.time()
while cola or any(corriendo.values()):
    for g in GPUS:
        for (p, lote) in list(corriendo[g]):
            if p.poll() is not None:
                corriendo[g].remove((p, lote))
                i, a, n = lote
                (hechos if p.returncode == 0 else fallidos).append(lote)
                print(f'[{time.time() - t0:7.0f} s] lote {i:3d} (mundos {a}–{a + n - 1}) en la GPU {g}: ' + ('listo' if p.returncode == 0 else f'FALLÓ ({p.returncode})'))
        while cola and len(corriendo[g]) < A_LA_VEZ_POR_GPU:
            i, a, n = lote = cola.popleft()
            p = correr(n, TICKS, PLAZAS, SEMILLA, a, f'{LOTES}/serie-{i:03d}.csv', f'{LOTES}/mundos-{i:03d}.csv', gpu=g,
                       log=f'{LOTES}/lote-{i:03d}.log', por_sector=POR_SECTOR, credito=CREDITO)
            corriendo[g].append((p, lote))
    time.sleep(2)
print(f'\n{len(hechos)} lotes listos, {len(fallidos)} fallidos · {time.time() - t0:.0f} s de pared para {MUNDOS} mundos × {TICKS} ticks')
for i, a, n in fallidos:
    print(f'\n── lote {i} (primerMundo {a}):'); print(''.join(open(f'{LOTES}/lote-{i:03d}.log').readlines()[-15:]))
# lo que midió cada lote
for i, a, n in sorted(hechos)[:4]:
    log = open(f'{LOTES}/lote-{i:03d}.log').read()
    print(f'lote {i}:', re.search(r'.*ms por tick.*', log).group(0))
    print('        ', re.search(r'invariantes.*', log).group(0)[:60])

## Un solo CSV, una fila por mundo

Se juntan los CSV de los lotes en el orden del número de mundo. Se comprueba que estén todos (0 … MUNDOS−1, sin repetir) y que todos los lotes cerraron con los invariantes en verde. Si el total pasa de 7 MB, se reparte en partes de hasta 7 MB, cada una con el encabezado.

In [ ]:
LIMITE = 7 * 1024 * 1024
for i, a, n in hechos:
    assert 'invariantes ✓' in open(f'{LOTES}/lote-{i:03d}.log').read(), f'el lote {i} no cerró con los invariantes en verde'
def filas_de(i):
    # un lote cabe siempre en un archivo, pero por si acaso también se leen sus partes
    rutas = [f'{LOTES}/mundos-{i:03d}.csv'] if os.path.exists(f'{LOTES}/mundos-{i:03d}.csv') else sorted(glob.glob(f'{LOTES}/mundos-{i:03d}-*.csv'))
    enc, filas = None, []
    for r in rutas:
        with open(r) as f:
            e = f.readline(); enc = enc or e
            filas += [x for x in f if x.strip()]
    return enc, filas
encabezado, todas = None, []
for i, a, n in sorted(hechos):
    e, f = filas_de(i)
    encabezado = encabezado or e
    assert e == encabezado
    todas += f
todas.sort(key=lambda x: int(x.split(',', 1)[0]))
numeros = [int(x.split(',', 1)[0]) for x in todas]
assert len(set(numeros)) == len(numeros), 'hay mundos repetidos'
faltan = sorted(set(range(MUNDOS)) - set(numeros))
print(f'{len(todas)} mundos juntados' + (f' · faltan {len(faltan)} (de lotes fallidos): {faltan[:5]}…' if faltan else ' · están todos'))

for v in glob.glob(f'{TRABAJO}/ecosim-g3-mundos*.csv'): os.remove(v)
total = len(encabezado) + sum(len(x) for x in todas)
if total <= LIMITE:
    salidas = [f'{TRABAJO}/ecosim-g3-mundos.csv']
    open(salidas[0], 'w').write(encabezado + ''.join(todas))
else:
    salidas, actual, largo = [], [], len(encabezado)
    for x in todas:
        if largo + len(x) > LIMITE:
            salidas.append(actual); actual, largo = [], len(encabezado)
        actual.append(x); largo += len(x)
    salidas.append(actual)
    partes = salidas; salidas = []
    for j, filas in enumerate(partes, 1):
        r = f'{TRABAJO}/ecosim-g3-mundos-{j}.csv'
        open(r, 'w').write(encabezado + ''.join(filas)); salidas.append(r)
for r in salidas:
    print(f'{r}: {os.path.getsize(r) / 1024 / 1024:.2f} MB · {sum(1 for _ in open(r)) - 1} mundos')

# la serie cada 10 minutos (los 20 primeros mundos de cada lote), también en uno
import pandas as pd
serie = pd.concat([pd.read_csv(f'{LOTES}/serie-{i:03d}.csv') for i, a, n in sorted(hechos)], ignore_index=True)
serie.to_csv(f'{TRABAJO}/ecosim-g3-serie.csv', index=False)
print(f'serie: {serie["economia"].nunique()} mundos × {serie["tick"].nunique()} instantes → ecosim-g3-serie.csv ({os.path.getsize(f"{TRABAJO}/ecosim-g3-serie.csv") / 1e6:.1f} MB)')

## Lo que dicen los miles de mundos

Las columnas del CSV de mundos: `mundo, semilla, poblacion0, poblacion, nacidos, muertes, muertesNinos, muertesHambre, muertesGuerra, contagios, picoEnfermos, edadMedia, saludMedia, saludMental, educacion`, el ánimo medio (`moral … estatus`, de 0 a 10000), los porcentajes (`conTecho, conPareja, pasandoHambre, amigos, enemigos`), la economía (`hoover, top10, movilidad, ipc, paro, empresas, quiebras, fundadas, indiceBolsa, escasez, trueques, fiados, impagos, ayudas, herencias, horasMonopolio, aranceles, evasiones`), lo social y la política (`robos … espionajes, estadosSinTerritorio, aliados, enemistados, duenosTerritorios`), la cultura (`estadosConEscritura, bibliotecas, obras, innovaciones, tecnologiaMax, tecnologiaMedia, modas`), el mundo (`sequias, plagas, desastres, bosque, contaminacion, suelo, acuifero, rutas, cohesion`) y la bolsa y los bancos de la v2 (`iliquidez, llamadasMargen, corridas, caidaMaxBolsa, prestamosVivos`). `duenosTerritorios` son seis dígitos: el Estado dueño de cada territorio.

Lo nuevo de la 3.1: `fundadasPersonas, empresasDePersonas, quiebrasPersonas, creditosEmprender, herenciasEmpresa` (los emprendedores), `innovacionesEmpresa, tecEmpresaMedia` (la investigación; 100 = la productividad de salida), `sueldoBase, pagoMedio, pagoMax` (monedas por año), `ventasAno1, ventasAnoFin, crecimientoReal` (unidades vendidas el segundo año y el último, y el crecimiento en %), `riquezaPersonasMiles, riquezaDuenos, riquezaResto` (la riqueza cuenta el dinero, la empresa propia y la parte del club), `pobresQueSalieron, ricosQueSalieron` (% de los que empezaron en el 20 % más pobre o más rico y ya no están ahí), `cambiosTrabajo` y `comprasOtroTerritorio` (el comercio).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
df = pd.concat([pd.read_csv(r) for r in salidas], ignore_index=True).copy()
df['crecimiento'] = (df['poblacion'] - df['poblacion0']) * 100 / df['poblacion0']
print(f'{len(df)} mundos · {TICKS // 3600} años cada uno\n')
print(df[['poblacion', 'crecimiento', 'nacidos', 'muertes', 'muertesHambre', 'hoover', 'paro', 'guerras', 'conquistas',
          'revoluciones', 'tecnologiaMax', 'bosque', 'indiceBolsa']].describe(percentiles=[.05, .5, .95]).round(1).T)
n = len(df)
def cuantos(m, txt): print(f'{txt:46s} {int(m.sum()):7d}  ({m.mean() * 100:5.1f} %)')
print()
cuantos(df['poblacion'] < df['poblacion0'], 'mundos que perdieron población')
cuantos(df['crecimiento'] < -20, 'mundos que perdieron más del 20 %')
cuantos(df['muertesHambre'] > 0, 'mundos con muertes de hambre o sed')
cuantos(df['guerras'] > 0, 'mundos con alguna guerra')
cuantos(df['conquistas'] > 0, 'mundos con alguna conquista')
cuantos(df['revoluciones'] > 0, 'mundos con alguna revolución')
cuantos(df['estadosSinTerritorio'] > 0, 'mundos con un Estado sin territorio')
cuantos(df['duenosTerritorios'].astype(str).str.zfill(6).map(lambda s: len(set(s)) == 1), 'mundos donde un Estado tiene los 6 territorios')
cuantos(df['estadosConEscritura'] == 3, 'mundos con escritura en los 3 Estados')
cuantos(df['caidaMaxBolsa'] >= 20, 'mundos con caída de bolsa ≥ 20 % en una hora')
cuantos(df['bosque'] < 5000, 'mundos con el bosque a menos de la mitad')
cuantos(df['crecimientoReal'] > 10, 'mundos cuya producción real creció más de un 10 %')
cuantos(df['fundadasPersonas'] >= 10, 'mundos con 10 o más empresas fundadas por personas')
cuantos(df['quiebrasPersonas'] > 0, 'mundos donde quebró la empresa de alguien')
print()
print(df[['crecimientoReal', 'fundadasPersonas', 'empresasDePersonas', 'innovacionesEmpresa', 'tecEmpresaMedia', 'sueldoBase',
          'pagoMedio', 'riquezaDuenos', 'riquezaResto', 'pobresQueSalieron', 'ricosQueSalieron']].describe(percentiles=[.05, .5, .95]).round(1).T)

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(16, 12))
def hist(a, col, titulo, color, bins=50, entero=False):
    x = df[col]
    if entero: a.hist(x, bins=range(int(x.min()), int(x.max()) + 2), color=color, align='left')
    else: a.hist(x, bins=bins, color=color)
    a.set_title(titulo)
hist(ax[0, 0], 'crecimiento', 'crecimiento de la población (%)', '#2a6f97')
hist(ax[0, 1], 'muertesHambre', 'muertes de hambre o sed por mundo', '#c0392b', entero=True)
hist(ax[0, 2], 'hoover', 'desigualdad (Hoover, %)', '#8e44ad')
hist(ax[1, 0], 'guerras', 'guerras por mundo', '#7f8c8d', entero=True)
hist(ax[1, 1], 'tecnologiaMax', 'tecnología máxima (100 = la de salida)', '#27ae60')
hist(ax[1, 2], 'indiceBolsa', 'índice de bolsa al final (salida = 100)', '#d35400')
hist(ax[2, 0], 'bosque', 'bosque medio (0–10000)', '#16a085')
hist(ax[2, 1], 'crecimientoReal', 'crecimiento de la producción real (%)', '#b9770e')
ax[2, 2].scatter(df['hoover'], df['crecimiento'], s=3, alpha=0.3, color='#2a6f97')
ax[2, 2].set_xlabel('Hoover (%)'); ax[2, 2].set_ylabel('crecimiento (%)'); ax[2, 2].set_title('desigualdad frente a crecimiento')
plt.tight_layout(); plt.show()

# el abanico de la población a lo largo de los años (los mundos que salen en la serie)
piv = serie.pivot(index='tick', columns='economia', values='poblacion')
anos = piv.index / 3600
q = piv.quantile([0.05, 0.25, 0.5, 0.75, 0.95], axis=1).T
plt.figure(figsize=(13, 5))
plt.fill_between(anos, q[0.05], q[0.95], color='#2a6f97', alpha=0.2, label='5–95 %')
plt.fill_between(anos, q[0.25], q[0.75], color='#2a6f97', alpha=0.4, label='25–75 %')
plt.plot(anos, q[0.5], color='#2a6f97', lw=2, label='mediana')
plt.xlabel('años simulados'); plt.ylabel('personas'); plt.title(f'{piv.shape[1]} mundos de la serie: el abanico de la población')
plt.legend(); plt.show()

## Qué hacer con esto

- **Más mundos**: sube `MUNDOS` (50.000 son 34 lotes; el CSV sale en 4 partes de ≤ 7 MB). Con `A_LA_VEZ_POR_GPU = 2` cada T4 usa ~3,3 GB; si `nvidia-smi` muestra la GPU por debajo del 90 % de uso, prueba con 3.
- **Más años**: `TICKS = 36000` son 10 años. El tiempo crece en proporción.
- **Mundos más pequeños**: `PLAZAS = 500` deja 6.250 mundos por lote y cuatro veces menos trabajo por mundo.
- **Reproducir un mundo**: el mundo `n` del CSV se repite exacto con `node motor/util/n-node.js --c1en ecosim/ecosim-g3.esl 1 21600 2000 20261018 - 6 1 n` (en `gpu`, `nativo` o `jit`: dan lo mismo, al bit).
- **Descargar**: los CSV quedan en `/kaggle/working` (panel *Output* → descargar). Para guardarlos con el cuaderno: *Save Version → Save & Run All*.